<!-- context: VAAET/notebooks/01_data_prep/data_preparation.ipynb
Module 1 of the VAAET pipeline — Traffic state classification training.
Consumes telemetry from Module 0 (traffic_data) and produces trained classifier.
ADR-008 documents the TF/Keras decision. Does not modify Module 0. -->

# Module 1 — Traffic State Classifier (Data Preparation)

Module 0 (Bootstrap) of VAAET (notebook `01_legacy_collection.ipynb`, now archived) detects vehicles, tracks them, and estimates their speed. Every minute it persists a record with average speed and counts by type in the `traffic_data` table on PostgreSQL. But those 9 raw fields do not answer the operational question that the SISE system needs: **what is the current traffic state on the bridge?**

This notebook implements the **intelligence layer** that transforms raw telemetry into a classification of 4 operational states:

| State | Code | Engineering Criteria |
|---|---|---|
| **Normal** | 0 | Free flow (default: everything that does not meet more severe criteria) |
| **Reduced** | 1 | Degraded flow: 5-40 km/h, 15-25 veh/min |
| **Congested** | 2 | Congestion: <5 km/h, >25 veh/min, persistence ≥2 min |
| **Accident** | 3 | Disruptive event: ~0 km/h after sudden braking (delta < -20 km/h), persistence ≥3 min |

**Architecture**: Tabular MLP with TensorFlow/Keras, designed to evolve to LSTM with temporal memory in a future iteration. See [ADR-008](../../docs/adr/ADR-008-tensorflow-keras-traffic-classifier.md) for the complete rationale.

In [ ]:
# Cell 0 — Environment Setup (Colab / Local)
#
# On Google Colab the CWD is /content, not the repo root.
# This cell clones or mounts the repo and %cd to the notebook
# directory so that relative paths (../../models, etc.)
# resolve correctly. On VS Code / local it is a no-op.

import os

try:
    import google.colab  # type: ignore[import-untyped]
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/titesen/vaaet.git"
    REPO_DIR = "/content/vaaet"
    NB_DIR = os.path.join(REPO_DIR, "notebooks", "01_data_prep")

    if not os.path.isdir(REPO_DIR):
        print("📦 Cloning VAAET repository...")
        os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    else:
        print("📂 Repository already present, updating...")
        os.system(f"git -C {REPO_DIR} pull --ff-only")

    os.chdir(NB_DIR)
    print(f"✅ Colab CWD → {os.getcwd()}")
else:
    print(f"✅ Local environment detected — CWD: {os.getcwd()}")

In [ ]:
# Cell 1 — Dependencies and Imports
#
# On Google Colab, TensorFlow is pre-installed. Only additional
# pipeline dependencies are installed here.

import subprocess
import sys

def install_if_missing(package: str, import_name: str | None = None) -> None:
    """Install a package if not available in the environment."""
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# Dependencies that may not be pre-installed
install_if_missing("imbalanced-learn", "imblearn")
install_if_missing("sqlalchemy")
install_if_missing("psycopg2-binary", "psycopg2")
install_if_missing("seaborn")
install_if_missing("joblib")

# Core
import numpy as np
import pandas as pd
import os
import getpass
from datetime import datetime

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# SMOTE
from imblearn.over_sampling import SMOTE

# SQLAlchemy
from sqlalchemy import create_engine, text

# Serialization
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Artifact paths
MODEL_DIR: str = os.path.join("..", "..", "models", "intelligence")
DATA_DIR: str = os.path.join("..", "..", "data", "processed")
RAW_DIR: str = os.path.join("..", "..", "data", "raw")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

print(f"✅ Dependencies loaded")
print(f"   TensorFlow {tf.__version__} | Pandas {pd.__version__} | NumPy {np.__version__}")
print(f"   Global seed: {RANDOM_SEED}")
print(f"   Artifacts → {os.path.abspath(MODEL_DIR)}")

## Data Source — Module 0 Telemetry

Data comes from the `traffic_data` table in PostgreSQL (AWS RDS), produced by Module 0 (Bootstrap perception). Each record represents one minute of processed video with 9 fields: average speed, counts by vehicle type (car, truck, bus, motorcycle, bicycle), and total.

The real dataset contains ~2000 records from the backup `traffic_data.backup` (`pg_dump` format). To use this notebook you need:

1. **Option A** — Direct connection to the RDS instance where Module 0 ran
2. **Option B** — Restore the backup to a local PostgreSQL instance: `pg_restore -d vaaet data/raw/traffic_data.backup`

Credentials are obtained via environment variables (`DB_HOST`, `DB_PORT`, `DB_NAME`, `DB_USER`, `DB_PASSWORD`) or interactive input. They are never hardcoded or printed in outputs.

On successful DB load, a raw copy is saved to `data/raw/traffic_data_raw.csv` to allow future runs without a connection.

In [ ]:
# Cell 2 — DB Connection + Telemetry Extraction

RAW_CSV_PATH: str = os.path.join("..", "..", "data", "raw", "traffic_data_raw.csv")


def get_db_config() -> dict[str, str]:
    """Get DB configuration from env vars or interactive input."""
    config = {
        "host": os.environ.get("DB_HOST", ""),
        "port": os.environ.get("DB_PORT", "5432"),
        "dbname": os.environ.get("DB_NAME", ""),
        "user": os.environ.get("DB_USER", ""),
        "password": os.environ.get("DB_PASSWORD", ""),
    }
    if not config["host"]:
        print("📋 PostgreSQL configuration (environment variables not found)")
        config["host"] = input("   Host: ").strip()
        config["port"] = input("   Port [5432]: ").strip() or "5432"
        config["dbname"] = input("   Database: ").strip()
        config["user"] = input("   User: ").strip()
        config["password"] = getpass.getpass("   Password: ")
    return config


def load_telemetry(config: dict[str, str]) -> pd.DataFrame:
    """Load raw telemetry from traffic_data via SQLAlchemy."""
    conn_str = (
        f"postgresql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['dbname']}"
    )
    engine = create_engine(conn_str)

    query = """
        SELECT id, clip_id, record_time, avg_speed,
               count_car, count_truck, count_bus,
               count_motorcycle, count_bicycle, total_vehicles
        FROM traffic_data
        ORDER BY record_time
    """
    df = pd.read_sql(text(query), engine)
    engine.dispose()
    return df


# Execution
try:
    db_config = get_db_config()
    df_raw = load_telemetry(db_config)
    # Save raw copy for future fallback (only the 10 original columns)
    df_raw.to_csv(RAW_CSV_PATH, index=False)
    print(f"✅ Telemetry loaded: {df_raw.shape[0]} records, {df_raw.shape[1]} columns")
    print(f"   Time range: {df_raw['record_time'].min()} → {df_raw['record_time'].max()}")
    print(f"   Raw CSV saved → {os.path.abspath(RAW_CSV_PATH)}")
    print(f"\n📊 Statistical summary:")
    display(df_raw.describe().round(2)) if "display" in dir() else print(df_raw.describe().round(2))
except Exception as e:
    print(f"🔴 DB connection error: {e}")
    print("   Attempting to load from raw CSV as fallback...")
    if os.path.exists(RAW_CSV_PATH):
        df_raw = pd.read_csv(RAW_CSV_PATH)
        print(f"✅ Raw CSV loaded: {df_raw.shape[0]} records")
    else:
        raise RuntimeError(
            "No DB connection or local raw CSV available. "
            "Restore traffic_data.backup or configure credentials."
        )

## Feature Engineering — From 9 Raw Fields to 14 Features

Raw telemetry (speed + counts) does not capture relationships between consecutive records or temporal patterns. Feature engineering expands the 9 original fields into 14 variables the model can exploit:

| Feature | Origin | Domain Justification |
|---|---|---|
| `avg_speed` | Direct | Primary indicator of vehicular flow |
| `total_vehicles` | Direct | Absolute traffic volume |
| `count_car` ... `count_bicycle` | Direct (5) | Vehicle composition — trucks and buses impact flow differently than cars |
| `heavy_vehicle_ratio` | Derived | Heavy vehicle proportion — heavy traffic degrades flow more |
| `delta_speed` | Derived (diff) | Acceleration/deceleration between consecutive minutes |
| `delta_count` | Derived (diff) | Volume change rate — detects accumulation |
| `transition_flag` | Derived | Binary signal: simultaneous sharp changes in speed and volume |
| `speed_variance` | Derived (rolling) | Recent variability — unstable vs stable traffic |
| `hour_of_day` | Temporal | Circadian traffic patterns (rush hour, nighttime) |
| `weather_condition` | Simulated | Environmental condition proxy based on hour (nighttime=risk) |

Derived features (`delta_*`, `speed_variance`) introduce NaN in the first records, which are dropped.

In [ ]:
# Cell 3 — Feature Engineering

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Transform 9 raw fields into 14 features for the classifier.

    Args:
        df: DataFrame with traffic_data columns.

    Returns:
        DataFrame with 14 features, no NaN.
    """
    out = df.copy()

    # Derived features
    # Heavy vehicle ratio (trucks + buses / total)
    out["heavy_vehicle_ratio"] = (
        (out["count_truck"] + out["count_bus"])
        / out["total_vehicles"].clip(lower=1)
    )

    # Inter-record deltas (acceleration / rate of change)
    out["delta_speed"] = out["avg_speed"].diff()
    out["delta_count"] = out["total_vehicles"].diff()

    # Transition flag: simultaneous sharp change in speed AND volume
    out["transition_flag"] = (
        (out["delta_speed"].abs() > 10) & (out["delta_count"].abs() > 5)
    ).astype(int)

    # Recent speed variability (5-minute window)
    out["speed_variance"] = out["avg_speed"].rolling(
        window=5, min_periods=1
    ).std()

    # Hour of day (circadian pattern)
    if pd.api.types.is_datetime64_any_dtype(out["record_time"]):
        out["hour_of_day"] = out["record_time"].dt.hour
    else:
        out["record_time"] = pd.to_datetime(out["record_time"])
        out["hour_of_day"] = out["record_time"].dt.hour

    # Simulated weather condition (hour-based proxy)
    # 0 = clear (6-18h), 1 = nighttime/risk (rest)
    out["weather_condition"] = (
        ~out["hour_of_day"].between(6, 18)
    ).astype(int)

    # Drop rows with NaN from diff()
    out = out.dropna(subset=["delta_speed", "delta_count"]).reset_index(drop=True)

    return out


# Execution
df_features = engineer_features(df_raw)

# Save features CSV for reproducibility (NOT to be used as raw data fallback)
csv_path = os.path.join(DATA_DIR, "traffic_telemetry.csv")
df_features.to_csv(csv_path, index=False)

print(f"✅ Features engineered: {df_features.shape[0]} records × {df_features.shape[1]} columns")
print(f"   Features CSV saved → {os.path.abspath(csv_path)}")
print(f"\n📊 Correlation with avg_speed:")
FEATURE_COLS: list[str] = [
    "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle", "count_bicycle",
    "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]
corr = df_features[FEATURE_COLS].corr()["avg_speed"].drop("avg_speed").sort_values()
print(corr.to_string())

## Auto-Labeling — Traffic Engineering Rules

Without manual annotation of thousands of records, we use engineering rules as a ground truth proxy. Thresholds are derived from bridge operation standards and domain experience:

- **Accident (3)** — most severe: speed ~0 km/h after sudden braking (`delta_speed < -20`) detected within a recent 5-minute window, sustained ≥3 consecutive records at <2 km/h. Assigned first so it is not overwritten.
- **Congested (2)**: speed <5 km/h with high density (>25 veh/min) sustained ≥2 records.
- **Reduced (1)**: speed between 5-40 km/h with moderate density (15-25 veh/min). Since each record is 1 minute, persistence >60s is already implicit.
- **Normal (0)**: everything else (default catch-all).

**Known limitation**: these labels are NOT human ground truth. A SISE operator corrects this via HITL (fields `is_human_validated` and `human_override_state` in the `traffic_classifications` table) in the Module 2 feedback loop. See [BIAS_AND_LIMITATIONS.md](../../docs/BIAS_AND_LIMITATIONS.md).

In [ ]:
# Cell 4 — Auto-Labeling + Class Distribution

STATE_LABELS: dict[int, str] = {
    0: "Normal",
    1: "Reduced",
    2: "Congested",
    3: "Accident",
}


def assign_traffic_state(df: pd.DataFrame) -> pd.Series:
    """Assign traffic states using engineering rules.

    Evaluation order: most severe first (Accident → Congested → Reduced).
    Normal is the default (code 0).

    Accident detection uses a two-phase model:
      1. Impact: sudden braking (delta_speed < -20) within a recent
         window of 5 records.
      2. Persistence: speed < 2 km/h sustained ≥ 3 consecutive records.

    Args:
        df: DataFrame with engineered features.

    Returns:
        Series with state codes (0-3).
    """
    states = pd.Series(0, index=df.index, dtype=int)  # Default: Normal

    # Accident (3): recent impact + sustained ~0 speed
    low_speed = df["avg_speed"] < 2
    # Phase 1 — Was there sudden braking in the last 5 minutes?
    braking = df["delta_speed"] < -20
    had_recent_braking = braking.rolling(window=5, min_periods=1).max().astype(bool)
    # Phase 2 — Has speed been ~0 for ≥3 consecutive records?
    consecutive_low = low_speed.rolling(window=3, min_periods=3).sum() >= 3
    accident_mask = low_speed & had_recent_braking & consecutive_low
    states[accident_mask] = 3

    # Congested (2): sustained congestion
    congestion = (df["avg_speed"] < 5) & (df["total_vehicles"] > 25)
    consecutive_congestion = congestion.rolling(window=2, min_periods=2).sum() >= 2
    stuck_mask = congestion & consecutive_congestion & (states != 3)
    states[stuck_mask] = 2

    # Reduced (1): degraded flow
    reduced_mask = (
        df["avg_speed"].between(5, 40)
        & df["total_vehicles"].between(15, 25)
        & (states == 0)  # Only if not Accident or Congested
    )
    states[reduced_mask] = 1

    return states


# Execution
df_features["traffic_state"] = assign_traffic_state(df_features)

# Distribution
dist = df_features["traffic_state"].value_counts().sort_index()
print("📊 Traffic state distribution:")
for code, count in dist.items():
    pct = 100 * count / len(df_features)
    print(f"   {STATE_LABELS[code]:>10} ({code}): {count:>5} records ({pct:.1f}%)")

# Verify at least 2 classes exist
n_classes = dist.index.nunique()
if n_classes < 2:
    print("🔴 Only 1 class found. Thresholds do not discriminate in this dataset.")
else:
    print(f"\n✅ {n_classes} classes detected")

# Classes without samples
for code, label in STATE_LABELS.items():
    if code not in dist.index:
        print(f"⚠️  Class '{label}' ({code}) has no samples — will be excluded from training")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71", "#f39c12", "#e74c3c", "#8e44ad"]
bars = ax.bar(
    [STATE_LABELS[c] for c in sorted(dist.index)],
    [dist[c] for c in sorted(dist.index)],
    color=[colors[c] for c in sorted(dist.index)],
)
ax.set_ylabel("Records")
ax.set_title("Traffic State Distribution (Auto-Labeling)")
for bar, count in zip(bars, [dist[c] for c in sorted(dist.index)]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(count), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## Balancing and Partitioning — SMOTE + Stratification

The real dataset is strongly imbalanced: ~80% Normal expected, with Accident potentially <1%. Training a model directly would produce a classifier that ignores minority classes.

**Strategy**:
1. **StandardScaler**: Normalizes features to mean=0, std=1 (required for neural networks)
2. **Train/Test split** (80/20): Stratified to maintain original proportions in both sets
3. **SMOTE** (Synthetic Minority Over-sampling Technique): Applied **only to the training set** to generate synthetic samples from minority classes. The test set remains intact as a realistic evaluation

The scaler is exported as an artifact (`feature_scaler.joblib`) so that production inference uses the same transformation.

In [ ]:
# Cell 5 — SMOTE + Train/Test Split

# Prepare matrices
X = df_features[FEATURE_COLS].values
y = df_features["traffic_state"].values

# Scaling
scaler = StandardScaler()

# Stratified Train/Test split
# Guard: stratify fails if any class has <2 samples
class_counts = np.bincount(y)
min_samples_per_class = class_counts[class_counts > 0].min()
can_stratify = min_samples_per_class >= 2

if can_stratify:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED
    )
else:
    print("⚠️ Class with <2 samples — split without stratification")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )

# Fit scaler ONLY on training, transform both
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"📊 Original partition:")
print(f"   Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
print(f"   Train distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

# SMOTE (training only)
# Verify minority class has at least k_neighbors+1 samples
train_counts = np.bincount(y_train)
min_class_count = train_counts[train_counts > 0].min()
k_neighbors = min(5, min_class_count - 1) if min_class_count > 1 else 1

if min_class_count < 2:
    print("⚠️ Class with <2 samples in train. SMOTE disabled — proceeding without balancing.")
    X_train_res, y_train_res = X_train, y_train
else:
    sm = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_neighbors)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
    print(f"\n✅ SMOTE applied (k_neighbors={k_neighbors}):")
    print(f"   Balanced train: {X_train_res.shape[0]} samples")
    print(f"   Distribution: {dict(zip(*np.unique(y_train_res, return_counts=True)))}")

# Export scaler
scaler_path = os.path.join(MODEL_DIR, "feature_scaler.joblib")
joblib.dump(scaler, scaler_path)
print(f"\n💾 Scaler saved → {os.path.abspath(scaler_path)}")

## Model Architecture — Tabular MLP

The model is a **Multi-Layer Perceptron (MLP)** implemented with `tf.keras.Sequential`. The architecture is deliberately simple — designed to validate the complete pipeline. A future iteration may evolve to LSTM with temporal memory.

**Why these dimensions?**
- **Dense(64)**: Input layer with sufficient capacity to learn non-linear combinations of 14 features
- **Dense(32)**: Compression layer that forces more abstract representations
- **BatchNormalization**: Stabilizes and accelerates training by normalizing activations between layers
- **Dropout(0.3 → 0.2)**: Decreasing regularization — more aggressive near the input (where there is more redundancy)
- **Softmax(n_classes)**: Probability distribution over the 4 states

In [ ]:
# Cell 6 — Model Definition + Training

# Dynamic number of classes (may be <4 if some state has no samples)
n_classes: int = len(np.unique(y))
n_features: int = X_train_res.shape[1]

print(f"🏗️ Building MLP model: {n_features} features → {n_classes} classes")

# Architecture
model = Sequential([
    Input(shape=(n_features,)),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation="relu"),
    BatchNormalization(),
    Dropout(0.2),
    Dense(n_classes, activation="softmax"),
], name="traffic_state_classifier")

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

# Callbacks
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        patience=5,
        factor=0.5,
        min_lr=1e-6,
        verbose=1,
    ),
]

# Training
print("\n🚀 Starting training...")
history = model.fit(
    X_train_res,
    y_train_res,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1,
)

# Training visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(history.history["loss"], label="Train Loss")
ax1.plot(history.history["val_loss"], label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss During Training")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(history.history["accuracy"], label="Train Accuracy")
ax2.plot(history.history["val_accuracy"], label="Val Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy During Training")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_epoch = np.argmin(history.history["val_loss"]) + 1
print(f"\n✅ Training completed — best epoch: {best_epoch}")

## Evaluation — Classification Metrics

The key metrics for this classifier are:

- **F1-macro** ≥ 0.85: Unweighted average of per-class F1. Equally penalizes performance on rare classes (Accident) and frequent classes (Normal)
- **Per-class Recall** > 0: Especially for Accident — a recall of 0 would mean the model never detects this critical class
- **Confusion matrix**: Identifies systematic confusions (e.g., Normal↔Reduced is the most likely confusion pair)

The model is exported as `.keras` (native standalone format) along with the label mapping.

In [ ]:
# Cell 7 — Evaluation + Model Export

# Predict on test set
y_proba = model.predict(X_test)
y_pred = y_proba.argmax(axis=1)

# Present class names
present_classes = sorted(np.unique(np.concatenate([y_test, y_pred])))
target_names = [STATE_LABELS[c] for c in present_classes]

# Classification Report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(
    y_test, y_pred,
    labels=present_classes,
    target_names=target_names,
    zero_division=0,
)
print(report)

# F1-macro
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
print(f"{'F1-macro':>15}: {f1_macro:.4f}")

if f1_macro >= 0.85:
    print(f"✅ F1-macro MEETS the target (≥ 0.85)")
else:
    print(f"⚠️  F1-macro BELOW target (≥ 0.85) — review balancing or thresholds")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=present_classes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — F1-macro: {f1_macro:.4f}")
plt.tight_layout()
plt.show()

# Per-class recall
print("\n📊 Per-class recall:")
for i, cls in enumerate(present_classes):
    row_sum = cm[i].sum()
    recall = cm[i, i] / row_sum if row_sum > 0 else 0.0
    status = "✅" if recall > 0 else "🔴"
    print(f"   {status} {STATE_LABELS[cls]:>10}: {recall:.4f}")

# Export model
model_path = os.path.join(MODEL_DIR, "traffic_classifier.keras")
model.save(model_path)

# Export label_mapping filtered to the classes the model actually saw
label_mapping = {c: STATE_LABELS[c] for c in present_classes}
label_path = os.path.join(MODEL_DIR, "label_mapping.joblib")
joblib.dump(label_mapping, label_path)

print(f"\n💾 Artifacts exported:")
print(f"   Model  → {os.path.abspath(model_path)} ({os.path.getsize(model_path) / 1024:.1f} KB)")
print(f"   Labels → {os.path.abspath(label_path)} (classes: {list(label_mapping.values())})")
print(f"   Scaler → {os.path.abspath(os.path.join(MODEL_DIR, 'feature_scaler.joblib'))}")

## Persistence — Two New Tables with FK

The classifier results are persisted in PostgreSQL with full traceability:

```
traffic_data (legacy, untouchable)
    ↓ FK: source_record_id
telemetry_raw (14 engineered features)
    ↓ FK: telemetry_id
traffic_classifications (prediction + HITL)
```

- **`telemetry_raw`**: Stores the 14 calculated features, with FK to the original record in `traffic_data`. Enables training reproducibility and auditing which data fed each prediction.
- **`traffic_classifications`**: Stores the model prediction, confidence, model version, and HITL fields (`is_human_validated`, `human_override_state`, `validated_at`) that will be activated when a SISE operator can confirm/dismiss alerts.

Persistence is **optional**: if no DB is configured, the notebook works completely and exports artifacts locally.

In [ ]:
# Cell 8 — Create Tables + Persist Results

DDL_TELEMETRY_RAW: str = """
CREATE TABLE IF NOT EXISTS telemetry_raw (
    id SERIAL PRIMARY KEY,
    source_record_id INTEGER REFERENCES traffic_data(id),
    record_time TIMESTAMP NOT NULL,
    avg_speed NUMERIC(5,2),
    total_vehicles INTEGER,
    count_car INTEGER,
    count_truck INTEGER,
    count_bus INTEGER,
    count_motorcycle INTEGER,
    count_bicycle INTEGER,
    heavy_vehicle_ratio NUMERIC(5,4),
    delta_speed NUMERIC(6,2),
    delta_count INTEGER,
    transition_flag SMALLINT DEFAULT 0,
    speed_variance NUMERIC(6,2),
    hour_of_day SMALLINT,
    weather_condition SMALLINT DEFAULT 0,
    UNIQUE (source_record_id)
);
"""

DDL_TRAFFIC_CLASSIFICATIONS: str = """
CREATE TABLE IF NOT EXISTS traffic_classifications (
    id SERIAL PRIMARY KEY,
    telemetry_id INTEGER REFERENCES telemetry_raw(id),
    classified_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    traffic_state SMALLINT NOT NULL,
    state_label TEXT NOT NULL,
    confidence NUMERIC(5,4) NOT NULL,
    model_version TEXT NOT NULL,
    is_human_validated BOOLEAN DEFAULT FALSE,
    human_override_state SMALLINT,
    validated_at TIMESTAMP,
    UNIQUE (telemetry_id, model_version)
);
"""

MODEL_VERSION: str = "mlp-v1.0"

TELEMETRY_COLS: list[str] = [
    "source_record_id", "record_time", "avg_speed", "total_vehicles",
    "count_car", "count_truck", "count_bus", "count_motorcycle",
    "count_bicycle", "heavy_vehicle_ratio", "delta_speed", "delta_count",
    "transition_flag", "speed_variance", "hour_of_day", "weather_condition",
]

INSERT_TELEMETRY_SQL: str = """
    INSERT INTO telemetry_raw (
        source_record_id, record_time, avg_speed, total_vehicles,
        count_car, count_truck, count_bus, count_motorcycle,
        count_bicycle, heavy_vehicle_ratio, delta_speed, delta_count,
        transition_flag, speed_variance, hour_of_day, weather_condition
    ) VALUES (
        :source_record_id, :record_time, :avg_speed, :total_vehicles,
        :count_car, :count_truck, :count_bus, :count_motorcycle,
        :count_bicycle, :heavy_vehicle_ratio, :delta_speed, :delta_count,
        :transition_flag, :speed_variance, :hour_of_day, :weather_condition
    )
    ON CONFLICT (source_record_id) DO NOTHING
"""

INSERT_CLASSIFICATION_SQL: str = """
    INSERT INTO traffic_classifications (
        telemetry_id, traffic_state, state_label,
        confidence, model_version
    ) VALUES (
        :telemetry_id, :traffic_state, :state_label,
        :confidence, :model_version
    )
    ON CONFLICT (telemetry_id, model_version) DO NOTHING
"""


def persist_results(
    df: pd.DataFrame,
    model: tf.keras.Model,
    scaler: StandardScaler,
    config: dict[str, str],
    feature_cols: list[str],
) -> None:
    """Create tables and persist features + classifications to PostgreSQL.

    Uses batch inserts (executemany via conn.execute with list of dicts)
    instead of row-by-row for better performance.

    Args:
        df: DataFrame with engineered features and traffic_state.
        model: Trained Keras model.
        scaler: Fitted StandardScaler.
        config: DB credentials.
        feature_cols: List of feature names.
    """
    conn_str = (
        f"postgresql://{config['user']}:{config['password']}"
        f"@{config['host']}:{config['port']}/{config['dbname']}"
    )
    engine = create_engine(conn_str)

    with engine.begin() as conn:
        # Create tables
        conn.execute(text(DDL_TELEMETRY_RAW))
        conn.execute(text(DDL_TRAFFIC_CLASSIFICATIONS))
        print("✅ Tables created (or already existed)")

        # Prepare telemetry_raw data
        df_telemetry = df.rename(columns={"id": "source_record_id"})[
            [c for c in TELEMETRY_COLS if c in df.columns or c == "source_record_id"]
        ].copy()

        # Ensure correct types for PostgreSQL
        df_telemetry["delta_count"] = df_telemetry["delta_count"].astype(int)
        df_telemetry["transition_flag"] = df_telemetry["transition_flag"].astype(int)
        df_telemetry["hour_of_day"] = df_telemetry["hour_of_day"].astype(int)
        df_telemetry["weather_condition"] = df_telemetry["weather_condition"].astype(int)

        # Insert telemetry_raw in batch
        telemetry_records = df_telemetry.to_dict(orient="records")
        result = conn.execute(text(INSERT_TELEMETRY_SQL), telemetry_records)
        print(f"📊 telemetry_raw: {len(telemetry_records)} records sent")

        # Classify the entire dataset
        X_all = scaler.transform(df[feature_cols].values)
        proba_all = model.predict(X_all, verbose=0)
        pred_all = proba_all.argmax(axis=1)
        conf_all = proba_all.max(axis=1)

        # Get telemetry_raw IDs
        telemetry_ids = conn.execute(
            text("SELECT id, source_record_id FROM telemetry_raw ORDER BY id")
        ).fetchall()
        source_to_telemetry = {row[1]: row[0] for row in telemetry_ids}

        # Prepare classification data in batch
        classification_records: list[dict] = []
        for idx, (_, row) in enumerate(df.iterrows()):
            source_id = row.get("id")
            telemetry_id = source_to_telemetry.get(source_id)
            if telemetry_id is None:
                continue
            classification_records.append({
                "telemetry_id": int(telemetry_id),
                "traffic_state": int(pred_all[idx]),
                "state_label": STATE_LABELS[int(pred_all[idx])],
                "confidence": float(round(conf_all[idx], 4)),
                "model_version": MODEL_VERSION,
            })

        if classification_records:
            conn.execute(text(INSERT_CLASSIFICATION_SQL), classification_records)

        print(f"📊 traffic_classifications: {len(classification_records)} records sent")

    engine.dispose()

    # Summary
    print(f"\n✅ Persistence completed (model_version={MODEL_VERSION})")
    dist = pd.Series(pred_all).value_counts().sort_index()
    for code, count in dist.items():
        print(f"   {STATE_LABELS[code]:>10}: {count} classifications")


# Execution
try:
    persist_results(df_features, model, scaler, db_config, FEATURE_COLS)
except NameError:
    print("⚠️ No DB configuration — results are local only")
    print("   Model artifacts are available at:")
    print(f"   {os.path.abspath(MODEL_DIR)}")
except Exception as e:
    print(f"🔴 Persistence error: {e}")
    print("   Model and artifacts are available locally")